# Final AI/ML Assignment — Kaggle GPU Notebook

This notebook runs the three assignment tasks sequentially:
1. **Task 1 — LoRA fine-tuning:** Qwen2.5-0.5B-Instruct for calendar-request JSON extraction, with base-vs-adapter evaluation on a held-out test set.
2. **Task 2 — RAG from scratch:** NIST AI RMF 1.0 PDF, sentence-transformer embeddings, FAISS or NumPy cosine retrieval, 150/300-word chunk comparison and recall@5.
3. **Task 3 — Inference benchmark:** FP16 vs 4-bit NF4 on the same 10 prompts.

## Kaggle setup
- Attach the dataset `deepsen839/kaggle-ai-ml-assignment-kaggle-fixed`.
- Enable **GPU** and **Internet** in Notebook options.
- Run top to bottom. Outputs are written to `/kaggle/working/assignment_results/`.
- The notebook removes the incompatible optional `torchao` package before importing PEFT (the assignment uses standard LoRA, not TorchAO), includes an idempotent Task 1 JSON-serialization patch, a Task 2 PDF downloader with official mirror fallbacks for NIST HTTP 403 errors, and skips a task if its expected outputs already exist. This prevents accidentally repeating completed training when resuming in the same Kaggle session.

**Important:** This notebook expects the repository source files in the attached Kaggle dataset. It does not claim the run is complete until all three task output folders are populated.

## 1. Check Kaggle runtime

In [ ]:
!nvidia-smi

import os, sys, platform, torch
print("Python:", sys.version)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU is not enabled. Open Notebook options → Accelerator → GPU, then rerun.")
print("GPU(s):", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## 2. Locate and copy the repository into writable storage

In [ ]:
from pathlib import Path
import shutil, zipfile

INPUT_ROOT = Path("/kaggle/input")
DATASET_ROOT = Path("/kaggle/input/datasets/deepsen839/kaggle-ai-ml-assignment-kaggle-fixed")
WORK_ROOT = Path("/kaggle/working")
TARGET = WORK_ROOT / "kaggle_ai_ml_assignment"

repo_source = None
if DATASET_ROOT.exists():
    direct = DATASET_ROOT / "kaggle_ai_ml_assignment"
    if (direct / "run_all.py").exists():
        repo_source = direct
    elif (DATASET_ROOT / "run_all.py").exists():
        repo_source = DATASET_ROOT
    else:
        matches = list(DATASET_ROOT.rglob("run_all.py"))
        if matches:
            repo_source = matches[0].parent
    if repo_source is None:
        for archive in DATASET_ROOT.rglob("*.zip"):
            with zipfile.ZipFile(archive) as zf:
                zf.extractall(WORK_ROOT)
            matches = list(WORK_ROOT.rglob("run_all.py"))
            if matches:
                repo_source = matches[0].parent
                break

if repo_source is None:
    # Fallback for Kaggle datasets mounted under a slightly different folder name.
    for candidate in INPUT_ROOT.rglob("run_all.py"):
        if "kaggle_ai_ml_assignment" in str(candidate.parent):
            repo_source = candidate.parent
            break

if repo_source is None:
    raise FileNotFoundError(
        f"Could not find run_all.py under {INPUT_ROOT}. Attach the assignment repository dataset."
    )

if TARGET.exists():
    shutil.rmtree(TARGET)
shutil.copytree(repo_source, TARGET, ignore=shutil.ignore_patterns("__pycache__", "*.pyc"))
print("Repository copied to:", TARGET)
print("Files:")
for p in sorted(TARGET.rglob("*")):
    if p.is_file(): print(p.relative_to(TARGET))

## 3. Install missing dependencies only

**Compatibility note:** Kaggle may preinstall `torchao 0.10.0`, which PEFT 0.21.2 rejects. The dependency cell removes this unused optional package before PEFT is imported. Run from the beginning in a fresh/restarted session.

In [ ]:
import importlib.util, subprocess

checks = {
    "fitz": "pymupdf",
    "sentence_transformers": "sentence-transformers",
    "peft": "peft",
    "transformers": "transformers",
    "bitsandbytes": "bitsandbytes",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
}
missing = [package for module, package in checks.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already installed.")


# PEFT 0.21.2's optional TorchAO dispatcher rejects the preinstalled torchao 0.10.0.
# This assignment uses ordinary LoRA on an unquantized base model, so TorchAO is not needed.
# Remove the incompatible optional package BEFORE importing PEFT.
import importlib
torchao_spec = importlib.util.find_spec("torchao")
if torchao_spec is not None:
    print("Removing incompatible optional torchao package before importing PEFT...")
    uninstall = subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
        text=True, capture_output=True
    )
    print((uninstall.stdout or uninstall.stderr)[-2000:])
    importlib.invalidate_caches()
    if importlib.util.find_spec("torchao") is not None:
        raise RuntimeError(
            "torchao is still importable after uninstall. Restart the Kaggle session "
            "and run this notebook from the beginning."
        )
else:
    print("No torchao package detected; standard PEFT LoRA path will be used.")

try:
    import faiss
    print("FAISS available")
except Exception as exc:
    print("FAISS unavailable; Task 2 will use the NumPy exact-search fallback:", repr(exc))

## 4. Verify repository and environment

In [ ]:
import transformers, peft, pandas, fitz, bitsandbytes
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Pandas:", pandas.__version__)
print("bitsandbytes:", bitsandbytes.__version__)

required = [TARGET / "task1_lora/train.py", TARGET / "task2_rag/rag.py", TARGET / "task3_inference/benchmark.py"]
for f in required:
    if not f.exists(): raise FileNotFoundError(f)
print("Repository preflight passed.")

RESULTS = WORK_ROOT / "assignment_results"
RESULTS.mkdir(parents=True, exist_ok=True)

## 5. Task 1 — LoRA fine-tuning

- Base model: `Qwen/Qwen2.5-0.5B-Instruct`
- Synthetic data split: 1,300 train / 100 validation / 100 held-out test
- LoRA: rank 8, alpha 16, dropout 0.05; attention and MLP projection modules
- Metrics: valid JSON rate and exact-match accuracy on the same held-out examples

### Compatibility patch
The earlier Kaggle run completed training and saved the model/evaluation artifacts, but crashed when printing a Python `set` inside `summary`. This cell patches **both** JSON serialization sites (`json.dump` and `json.dumps`) before Task 1 runs. It also skips training when the complete Task 1 artifacts are already present.

In [ ]:
from pathlib import Path
import json, subprocess, sys

TASK1 = TARGET / "task1_lora/train.py"
source = TASK1.read_text()

# Patch saving and console-print serialization. Safe to run repeatedly.
source = source.replace(
    "json.dump(summary, f, indent=2)",
    "json.dump(summary, f, indent=2, default=lambda x: sorted(x) if isinstance(x, set) else str(x))"
)
source = source.replace(
    "print(json.dumps(summary, indent=2))",
    "print(json.dumps(summary, indent=2, default=lambda x: sorted(x) if isinstance(x, set) else str(x)))"
)
TASK1.write_text(source)
print("Task 1 serialization compatibility patch applied.")

# Defensive guard: never let the known-incompatible TorchAO 0.10.0 reach PEFT 0.21.2.
# Cell 6 should already have removed it before PEFT was imported.
import importlib.util
if importlib.util.find_spec("torchao") is not None:
    raise RuntimeError(
        "Incompatible optional torchao package is still installed. "
        "Restart the Kaggle session and run cells from the beginning; "
        "the dependency cell removes torchao before importing PEFT."
    )

TASK1_OUT = RESULTS / "task1"
required_task1 = [
    TASK1_OUT / "base_test_results.json",
    TASK1_OUT / "finetuned_test_results.json",
    TASK1_OUT / "summary.json",
    TASK1_OUT / "failure_cases.json",
    TASK1_OUT / "lora_adapter" / "adapter_config.json",
]
if all(p.exists() for p in required_task1):
    print("Task 1 artifacts already complete. Skipping expensive 3-epoch retraining.")
else:
    print("Task 1 artifacts incomplete. Starting training...")
    subprocess.run([sys.executable, str(TASK1), "--epochs", "3"], cwd=TARGET, check=True)

missing = [str(p) for p in required_task1 if not p.exists()]
if missing:
    raise RuntimeError("Task 1 did not produce all required files: " + ", ".join(missing))
print("Task 1 verified. Files:")
for p in sorted(TASK1_OUT.rglob("*")):
    if p.is_file(): print(p.relative_to(TASK1_OUT), p.stat().st_size, "bytes")

## 6. Task 2 — RAG from scratch

The downloader validates any cached PDF and tries the NIST source plus official GovInfo and U.S. DOT ROSA-P mirrors if Kaggle receives an HTTP 403. Keep Kaggle Internet enabled. The notebook then extracts the PDF, builds 150- and 300-word chunks, embeds with `all-MiniLM-L6-v2`, and evaluates 15 questions using recall@5.

Downloads and extracts the public NIST AI RMF 1.0 PDF, builds 150- and 300-word chunks, embeds with `all-MiniLM-L6-v2`, and evaluates 15 questions using recall@5. Retrieval uses FAISS when available and exact NumPy cosine search otherwise. The selected retrieved passages are passed to Qwen2.5-0.5B-Instruct for grounded answer generation.

In [ ]:
import re
TASK2 = TARGET / "task2_rag/rag.py"

# Fix the NIST-host 403 seen in the previous Kaggle run.
# Validate cached PDFs, then try the NIST URL and two official government mirrors.
task2_source = TASK2.read_text()
downloader_patch = r"""
def download_pdf():
    import urllib.request
    from pathlib import Path

    pdf_path = Path(PDF)
    pdf_path.parent.mkdir(parents=True, exist_ok=True)

    # Reuse only a real PDF; remove truncated/HTML error responses.
    if pdf_path.exists():
        try:
            if pdf_path.read_bytes()[:5] == b"%PDF-":
                print(f"Using cached PDF: {pdf_path}")
                return pdf_path
        except Exception:
            pass
        pdf_path.unlink(missing_ok=True)

    candidates = [
        PDF_URL,
        "https://www.govinfo.gov/content/pkg/GOVPUB-C13-ef6882c6bd970bc733de394f469ea3f3/pdf/GOVPUB-C13-ef6882c6bd970bc733de394f469ea3f3.pdf",
        "https://rosap.ntl.bts.gov/view/dot/93262/dot_93262_DS1.pdf",
    ]
    errors = []
    for candidate_url in candidates:
        try:
            request = urllib.request.Request(
                candidate_url,
                headers={
                    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/124.0 Safari/537.36",
                    "Accept": "application/pdf,*/*",
                },
            )
            with urllib.request.urlopen(request, timeout=120) as response:
                payload = response.read()
            if not payload.startswith(b"%PDF-") or len(payload) < 100_000:
                raise ValueError("Downloaded response is not a valid-sized PDF")
            pdf_path.write_bytes(payload)
            print(f"Downloaded PDF from {candidate_url} ({len(payload):,} bytes)")
            return pdf_path
        except Exception as exc:
            errors.append(f"{candidate_url}: {type(exc).__name__}: {exc}")
            print("PDF source failed:", errors[-1])

    raise RuntimeError(
        "Could not download the NIST AI RMF PDF from any configured source. "
        "Check Kaggle Internet access. Details: " + " | ".join(errors)
    )
"""
pattern = r"def download_pdf\(\):.*?\n(?=def extract_pages\()"
patched_source, count = re.subn(pattern, downloader_patch.strip() + "\n\n", task2_source, count=1, flags=re.S)
if count != 1:
    raise RuntimeError("Could not locate download_pdf() in task2_rag/rag.py; repository version may have changed.")
TASK2.write_text(patched_source)
print("Task 2 downloader patched: cached-PDF validation + NIST/GovInfo/ROSA-P fallback.")

TASK2_OUT = RESULTS / "task2"
required_task2 = [
    TASK2_OUT / "test_questions_and_gold_passages.json",
    TASK2_OUT / "retrieval_150.json",
    TASK2_OUT / "retrieval_300.json",
    TASK2_OUT / "retrieval_comparison.csv",
    TASK2_OUT / "retrieval_failures.json",
    TASK2_OUT / "generated_answers.json",
]
if all(p.exists() for p in required_task2):
    print("Task 2 artifacts already complete. Skipping RAG rerun.")
else:
    subprocess.run([sys.executable, str(TASK2)], cwd=TARGET, check=True)

missing = [str(p) for p in required_task2 if not p.exists()]
if missing:
    raise RuntimeError("Task 2 did not produce all required files: " + ", ".join(missing))
print("Task 2 verified. Files:")
for p in sorted(TASK2_OUT.glob("*")):
    if p.is_file():
        print(p.name, p.stat().st_size, "bytes")


## 7. Task 3 — FP16 vs 4-bit inference benchmark

Runs the same 10 prompts with FP16 and 4-bit NF4, greedy decoding and a 128-token cap. It records aggregate tokens/second, peak CUDA allocated/reserved memory and per-prompt outputs. CUDA peak measurements are benchmark-reported peaks, not necessarily total system memory.

In [ ]:
TASK3 = TARGET / "task3_inference/benchmark.py"
TASK3_OUT = RESULTS / "task3"
required_task3 = [
    TASK3_OUT / "benchmark_summary.json",
    TASK3_OUT / "benchmark_summary.csv",
    TASK3_OUT / "prompt_outputs.json",
    TASK3_OUT / "prompt_outputs.csv",
]
if all(p.exists() for p in required_task3):
    print("Task 3 artifacts already complete. Skipping benchmark rerun.")
else:
    subprocess.run([sys.executable, str(TASK3)], cwd=TARGET, check=True)
missing = [str(p) for p in required_task3 if not p.exists()]
if missing:
    raise RuntimeError("Task 3 did not produce all required files: " + ", ".join(missing))
print("Task 3 verified. Files:")
for p in sorted(TASK3_OUT.glob("*")):
    if p.is_file(): print(p.name, p.stat().st_size, "bytes")

## 8. Review measured results

In [ ]:
import json, pandas as pd
from IPython.display import display

for task in ["task1", "task2", "task3"]:
    folder = RESULTS / task
    print("\n", "=" * 24, task.upper(), "=" * 24)
    if not folder.exists():
        print("No output folder found")
        continue
    for name in ["summary.json", "benchmark_summary.json", "retrieval_comparison.csv"]:
        path = folder / name
        if path.exists():
            print("\n", name)
            if path.suffix == ".json":
                print(json.dumps(json.loads(path.read_text()), indent=2)[:12000])
            else:
                display(pd.read_csv(path))

print("\nAll result files:")
for p in sorted(RESULTS.rglob("*")):
    if p.is_file(): print(p.relative_to(RESULTS), p.stat().st_size, "bytes")

## 9. Export results

In [ ]:
import zipfile
RESULT_ZIP = WORK_ROOT / "assignment_results.zip"
with zipfile.ZipFile(RESULT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in RESULTS.rglob("*"):
        if p.is_file(): zf.write(p, p.relative_to(WORK_ROOT))
print("Created:", RESULT_ZIP)
print("Size:", round(RESULT_ZIP.stat().st_size / 1024 / 1024, 2), "MB")

## 10. Completion checklist

- [ ] Task 1: base/fine-tuned results, summary, three actual failure examples and saved LoRA adapter.
- [ ] Task 2: 15 gold passages, retrieval results for both chunk sizes, recall@5 comparison, up to two actual retrieval failures and generated answers.
- [ ] Task 3: FP16 and 4-bit benchmark summaries and outputs for the same 10 prompts.
- [ ] Review model outputs manually and record quality observations.
- [ ] Complete the four-page report with actual measured results, failure analysis, 7B memory estimates and the 20-user serving plan.
- [ ] Include source code and report in the GitHub submission.

**A notebook run is complete only if all three task cells and the export cell finish successfully.** If a task cell fails, fix that task and rerun only that cell after checking which output artifacts were already produced.